In [16]:
import numpy as np
import pandas as pd

In [17]:
_ROOT_DIR = "./../../"

In [18]:
def call_data():
    ret_train_df = pd.read_csv(_ROOT_DIR+ './data/raw/cell2celltrain.csv')
    ret_test_df = pd.read_csv(_ROOT_DIR+ './data/raw/cell2cellholdout.csv')


    return ret_train_df, ret_test_df

In [19]:
# Churn (Yes/No) -> 1/0
# test_df에는 churn이 NaN로 되어있음, 수정 불필요
def preprocess_churn(train_df):
    train_df['Churn'] = train_df['Churn'].map({'Yes': 1, 'No': 0})

    return train_df

In [20]:
# 그 외 인코딩
def preprocess_encoding(train_df, test_df):
    # Yes/No -> 1/0
    binary_cols = [
    'ChildrenInHH', 'HandsetRefurbished', 'HandsetWebCapable', 'TruckOwner', 
    'RVOwner', 'BuysViaMailOrder', 'RespondsToMailOffers', 'OptOutMailings', 
    'NonUSTravel', 'OwnsComputer', 'HasCreditCard', 'NewCellphoneUser', 
    'NotNewCellphoneUser', 'OwnsMotorcycle', 'MadeCallToRetentionTeam'
    ]

    for col in binary_cols:
        train_df[col] = train_df[col].map({'Yes': 1, 'No': 0})
        test_df[col] = test_df[col].map({'Yes': 1, 'No': 0})

    
    # CreditRating
    credit_mapping = {
    '1-Highest': 1, '2-High': 2, '3-Good': 3, '4-Medium': 4, 
    '5-Low': 5, '6-VeryLow': 6, '7-Lowest': 7
    }
    train_df['CreditRating'] = train_df['CreditRating'].map(credit_mapping)
    test_df['CreditRating'] = test_df['CreditRating'].map(credit_mapping)

    # one-hot
    train_df = pd.get_dummies(train_df, columns=['PrizmCode', 'Occupation'], prefix=['Prizm', 'Occ'], dtype=int)
    test_df = pd.get_dummies(test_df, columns=['PrizmCode', 'Occupation'], prefix=['Prizm', 'Occ'], dtype=int)

    # HandsetPrice의 Unknown -> NaN으로 변환 후 중앙값으로 대체
    train_df['HandsetPrice'] = pd.to_numeric(train_df['HandsetPrice'].replace('Unknown', np.nan))
    test_df['HandsetPrice'] = pd.to_numeric(test_df['HandsetPrice'].replace('Unknown', np.nan))

    median_price = train_df['HandsetPrice'].median()
    train_df['HandsetPrice'] = train_df['HandsetPrice'].fillna(median_price)
    test_df['HandsetPrice'] = test_df['HandsetPrice'].fillna(median_price)

    # MaritalStatus의 Unknown -> NaN으로 변환 후 one-hot 인코딩
    train_df['MaritalStatus'] = train_df['MaritalStatus'].replace('Unknown', np.nan)
    test_df['MaritalStatus'] = test_df['MaritalStatus'].replace('Unknown', np.nan)
    train_df = pd.get_dummies(train_df, columns=['MaritalStatus'], prefix='Marital', dtype=int)
    test_df = pd.get_dummies(test_df, columns=['MaritalStatus'], prefix='Marital', dtype=int)


    return train_df, test_df
    

In [21]:
# Service area 제거
# 비슷한 값을 가지고 있는 PrizmCode가 존재
def preprocess_service_area(train_df, test_df):
    train_df = train_df.drop(columns=['ServiceArea'])
    test_df = test_df.drop(columns=['ServiceArea'])

    return train_df, test_df

In [22]:
# Homeownership 제거
# 필요없음
def preprocess_home_ownership(train_df, test_df):
    train_df = train_df.drop(columns=['Homeownership'])
    test_df = test_df.drop(columns=['Homeownership'])

    return train_df, test_df

In [23]:
def base_preprocess(train_df, test_df):
    train_df = preprocess_churn(train_df)
    train_df, test_df = preprocess_encoding(train_df, test_df)
    train_df, test_df = preprocess_service_area(train_df, test_df)
    train_df, test_df = preprocess_home_ownership(train_df, test_df)

    return train_df, test_df

In [24]:
def remove_null_rows(train_df, test_df, columns):
    ret_train_df = train_df.dropna(subset=columns)
    ret_test_df = test_df.dropna(subset=columns)
    return ret_train_df, ret_test_df

In [25]:
def remove_missing_values(train_df, test_df):
    
    cols = [['Handsets', 'HandsetModels', 'CurrentEquipmentDays'],
    ['PercChangeRevenues', 'PercChangeMinutes'],
    ['DirectorAssistedCalls', 'TotalRecurringCharge', 'RoamingCalls', 'OverageMinutes', 'MonthlyRevenue', 'MonthlyMinutes'],
    ['AgeHH1', 'AgeHH2']]

    for col in cols:
        train_df, test_df = remove_null_rows(train_df, test_df, col)

    return train_df, test_df

In [26]:
# 이후 모델 학습을 용이하게 하기 위해 'Churn'열을 제거하고 따로 저장 (모델 학습 시 X, y로 나누기 위해)
def preprocess_remove_churn(train_df, test_df):
    train_churn = train_df['Churn']
    train_df = train_df.drop(columns=['Churn'])
    test_df = test_df.drop(columns=['Churn'])

    return train_df, test_df, train_churn

In [27]:
def preprocess_final():
    train_df, test_df = call_data()
    train_df, test_df = base_preprocess(train_df, test_df)
    train_df, test_df = remove_missing_values(train_df, test_df)
    train_df, test_df, train_churn = preprocess_remove_churn(train_df, test_df)

    return train_df, test_df, train_churn

In [28]:
train_df, test_df, train_churn = preprocess_final()

In [29]:
display(train_df.head())
display(test_df.head())
display(train_churn.head())

,CustomerID,MonthlyRevenue,MonthlyMinutes,TotalRecurringCharge,DirectorAssistedCalls,OverageMinutes,RoamingCalls,PercChangeMinutes,PercChangeRevenues,DroppedCalls,...,Occ_Clerical,Occ_Crafts,Occ_Homemaker,Occ_Other,Occ_Professional,Occ_Retired,Occ_Self,Occ_Student,Marital_No,Marital_Yes
0,3000002,24.00,219.0,22.0,0.25,0.0,0.0,-157.0,-19.0,0.7,...,0,0,0,0,1,0,0,0,1,0
1,3000010,16.99,10.0,17.0,0.00,0.0,0.0,-4.0,0.0,0.3,...,0,0,0,0,1,0,0,0,0,1
2,3000014,38.00,8.0,38.0,0.00,0.0,0.0,-2.0,0.0,0.0,...,0,1,0,0,0,0,0,0,0,1
3,3000022,82.28,1312.0,75.0,1.24,0.0,0.0,157.0,8.1,52.0,...,0,0,0,1,0,0,0,0,1,0
4,3000026,17.14,0.0,17.0,0.00,0.0,0.0,0.0,-0.2,0.0,...,0,0,0,0,1,0,0,0,0,1


,CustomerID,MonthlyRevenue,MonthlyMinutes,TotalRecurringCharge,DirectorAssistedCalls,OverageMinutes,RoamingCalls,PercChangeMinutes,PercChangeRevenues,DroppedCalls,...,Occ_Clerical,Occ_Crafts,Occ_Homemaker,Occ_Other,Occ_Professional,Occ_Retired,Occ_Self,Occ_Student,Marital_No,Marital_Yes
0,3000006,57.49,483.0,37.0,0.25,23.0,0.0,532.0,51.0,8.3,...,0,0,0,1,0,0,0,0,1,0
1,3000018,55.23,570.0,72.0,0.00,0.0,0.0,38.0,0.0,9.7,...,0,0,0,0,1,0,0,0,1,0
2,3000034,97.34,1039.0,50.0,4.95,420.0,0.0,198.0,23.3,12.7,...,0,1,0,0,0,0,0,0,0,1
3,3000070,35.59,153.0,30.0,0.00,16.0,0.0,30.0,7.3,2.0,...,0,0,0,1,0,0,0,0,1,0
4,3000074,55.27,1213.0,50.0,0.74,0.0,1.3,169.0,1.0,2.7,...,0,0,0,1,0,0,0,0,1,0


0    1
1    1
2    0
3    0
4    1
Name: Churn, dtype: int64

In [30]:
train_df.to_csv(_ROOT_DIR + './data/preprocessed/cell2cell_train.csv', index=False)
test_df.to_csv(_ROOT_DIR + './data/preprocessed/cell2cell_test.csv', index=False)
train_churn.to_csv(_ROOT_DIR + './data/preprocessed/cell2cell_train_churn.csv', index=False)